In [8]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

In [9]:
remove_pitch_types = ['EP', 'FA', 'PO', 'IN', 'UN']
min_pitch_ct = 30
min_years = 3
keep_cols = ['release_speed', 'release_spin_rate', 'release_pos_x', 'release_pos_z', 'release_extension', 'spin_axis']
k_vals = [2,3,4,5,6,7,9,11]

In [10]:
def clean_data(filename):
    df = pd.read_csv(filename)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df['year'] = df['game_date'].dt.year.astype(int)
    df = df.sort_values('game_date').reset_index(drop=True)
    keep_cols = ['release_speed', 'release_spin_rate', 'release_pos_x', 'release_pos_z', 'release_extension', 'spin_axis', 'player_name', 'game_date', 'year', 'pitch_type', 'spin_axis']
    df = df.dropna(subset=keep_cols)
    return df


In [20]:
metrics = ['euclidean', 'manhattan']

def prep_data(df, pitcher_name):
    df = df[df['player_name'] == pitcher_name].copy()
    df = df[~df['pitch_type'].isin(remove_pitch_types)]
    count = df['pitch_type'].value_counts()
    valid_types = count[count >= min_pitch_ct].index
    df = df[df['pitch_type'].isin(valid_types)]

    if df.empty or df['pitch_type'].nunique() < 2:
        return None, None, None, None
    
    X = df[keep_cols].values
    y = df['pitch_type'].values
    years = df['year'].values
    return X, y, list(valid_types), years

def forward_chain(years):
    unique_years = sorted(np.unique(years))
    if len(unique_years) < min_years:
        return

    for i in range(1, len(unique_years)):
        train_years = unique_years[:i]
        test_year = unique_years[i]
        train_idx = np.where(np.isin(years, train_years))[0]
        test_idx = np.where(years == test_year)[0]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue
        yield train_idx, test_idx, train_years, test_year

def select_k(X, y, years, k_vals):
    fold = {k: [] for k in k_vals}
    best_params = {'k': k_vals[0], 'metric': metrics[0]}
    best_score = -1.0
    fold = {(k,m): [] for k in k_vals for m in metrics}

    splits = list(forward_chain(years))
    if not splits:
        split = int(len(X)*0.7)
        splits = [(np.arange(split), np.arange(split, len(X)), ['all'], 'holdout')]

    for k in k_vals:
        for metric in metrics:
            fold_scores = []
            for train_idx, test_idx, train_years, test_year in splits:
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                if len(np.unique(y_train)) < 2:
                    continue

                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('knn', KNeighborsClassifier(n_neighbors=k, metric=metric, weights='distance'))
                ])

                pipeline.fit(X_train, y_train)
                score = f1_score(y_test, pipeline.predict(X_test), average='weighted', zero_division=0)

                fold_scores.append(score)
                fold[(k, metric)].append({
                    'train_years': list(train_years),
                    'test_year': test_year,
                    'n_train': len(train_idx),
                    'n_test': len(test_idx),
                    'weighted_f1': round(score, 3)
                })

        if fold_scores:
            mean_score = np.mean(fold_scores)
            if mean_score > best_score:
                best_score = mean_score
                best_params = {'k': k, 'metric': metric}

    return best_params, fold

def select_k_kmeans(X, y, years, k_vals):
    fold = {k: [] for k in k_vals}
    best_k = None
    best_score = -1.0
    result = []

    splits = list(forward_chain(years))
    if not splits:
        split = int(len(X)*0.7)
        splits = [(np.arange(split), np.arange(split, len(X)), ['all'], 'holdout')]

    for k in k_vals:
        for train_idx, test_idx, train_years, test_year in splits:
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if len(np.unique(y_train)) < k:
                continue

            sc = StandardScaler()
            xTrainScale = sc.fit_transform(X_train)
            xTestScale = sc.transform(X_test)
            kMeansObj = KMeans(n_clusters = k).fit(xTrainScale)
            adjRand = adjusted_rand_score(y_test, kMeansObj.predict(xTestScale))
            fold[k].append(adjRand)

            result.append({
                'k': k,
                'train_years': str([int(y) for y in sorted(np.unique(years))]),
                'test_year': test_year,
                'n_train': len(train_idx),
                'n_test': len(test_idx),
                'adj_rand': round(adjRand, 3)
            })

        if fold[k]:
            meanVal = np.mean(fold[k])
            if meanVal > best_score:
                best_score = meanVal
                best_k = k
    return best_k, best_score, result


def train_and_eval(X, y, best_params):
    split = int(len(X)*0.7)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    if len(np.unique(y_train)) < 2 or len(X_test)==0:
        return None, None, None
    
    k = best_params['k']
    metric = best_params['metric']

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k, metric=metric, weights='distance'))
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    weighted_f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    report = classification_report(y_test, preds, zero_division=0)

    #adding kmeans clustering for adjusted rand
    # sc = StandardScaler()
    # xTrainScale = sc.fit_transform(X_train)
    # xTestScale = sc.transform(X_test)
    # kMeansObj = KMeans(n_clusters = len(np.unique(y_train))).fit(xTrainScale)
    # adjRand = adjusted_rand_score(y_test, kMeansObj.predict(xTestScale))

    return weighted_f1, report, pipeline

def run_kmeans_experiment(filepath, output_csv='kmeans_results.csv', folds_csv='kmeans_folds.csv'):
    df = clean_data(filepath)
    pitchers = df['player_name'].unique()
    res = []
    folds = []

    for pitcher in pitchers:
        X, y, pitch_types, years = prep_data(df, pitcher)
        if X is None:
            print(f'Skipping {pitcher}...not enough data or pitch types')
            continue

        n_years = len(np.unique(years))
        if n_years < min_years:
            print(f'Warning: {pitcher} only has data for {n_years} years')

        best_k, best_adj_rand, folds_data = select_k_kmeans(X, y, years, k_vals)

        if best_k is None:
            continue

        for row in folds_data:
            row['pitcher'] = pitcher
            folds.append(row)
        res.append({
            'pitcher': pitcher,
            'pitch_types': ','.join(pitch_types),
            #'years': ','.join(str(int(y)) for yr in years),
            'best_k': best_k,
            'best_adj_rand': round(best_adj_rand, 3),
            'n_years': n_years,
            'n_pitches': len(X)
        })
    summary_df = pd.DataFrame(res).sort_values('best_adj_rand', ascending=False)
    summary_df.to_csv(output_csv, index=False)

    folds_df = pd.DataFrame(folds)
    folds_df.to_csv(folds_csv, index = False)
    return summary_df


def run_experiment(filepath, output_csv='knn_results.csv', folds_csv='knn_folds.csv'):
    df = clean_data(filepath)
    pitchers = df['player_name'].unique()

    res = []
    all_folds = []

    for pitcher in pitchers:
        X, y, pitch_types, years = prep_data(df, pitcher)
        if X is None:
            print(f'Skipping {pitcher}...not enough data or pitch types')
            continue
        n_years = len(np.unique(years))
        if n_years < min_years:
            print(f'Warning: {pitcher} only has data for {n_years} years')
        best_params, fold_info = select_k(X, y, years, k_vals)
        weighted_f1, report, pipeline = train_and_eval(X, y, best_params)

        if weighted_f1 is None:
            print(f'Skipping {pitcher}...not enough data for training/testing')
            continue

        k = best_params['k']
        metric = best_params['metric']

        # print(f'\n{'='*55}')
        # print(f'Pitcher: {pitcher}')
        # print(f'Years of data: {[int(y) for y in sorted(np.unique(years))]}')
        # print(f'Pitch types: {pitch_types}')
        # print(f'Best k: {best_params["k"]}')
        # print(f'Weighted F1: {weighted_f1:.3f}')
        # print(f"\n{report}")

        # print(f'Forward-chain folds (k={best_params["k"]}):')
        for fold in fold_info[(k, metric)]:
            all_folds.append({
                'pitcher': pitcher,
                'test_year': fold['test_year'],
                'n_train': fold['n_train'],
                'n_test': fold['n_test'],
                'weighted_f1': fold['weighted_f1']
            })

            # train_str = str(fold['train_years'])
            # print(f'train={train_str} | test={fold["test_year"]} | n_train={fold["n_train"]} | n_test={fold["n_test"]} | weighted_f1={fold["weighted_f1"]}')
        res.append({
            'pitcher': pitcher,
            'pitch_types': ','.join(pitch_types),
            'years': str([int(y) for y in sorted(np.unique(years))]),
            'best_k': k,
            'best_metric': metric,
            'n_years': n_years,
            'n_pitches': len(X),
            'weighted_f1': round(weighted_f1, 3)
        })

        summary_df = pd.DataFrame(res).sort_values('weighted_f1', ascending=False)
        summary_df.to_csv(output_csv, index=False)
        # print(f'\nSummary saved to {output_csv}')
    
    pd.DataFrame(all_folds).to_csv(folds_csv, index=False)
    print(f'Fold details saved to {folds_csv}')
    
    return summary_df

In [21]:
# run_experiment('pitcher_data_detailed_cleaned.csv', 'knn_results.csv')
run_kmeans_experiment('pitcher_data_detailed_cleaned.csv')

,pitcher,pitch_types,best_k,best_adj_rand,n_years,n_pitches
35,"Miller, Mason","FF,SL,CH,FC",2,0.961,3,2512
34,"Pepiot, Ryan","FF,CH,SL,FC,CU,SI",5,0.836,4,5808
32,"Sears, JP","FF,ST,CH,SL,SI,CU",3,0.792,4,8866
9,"Gausman, Kevin","FF,FS,SL,CH,SI",3,0.737,10,26386
10,"Manaea, Sean","SI,FF,CH,SL,ST,FC",3,0.719,10,19398
1,"Sale, Chris","SL,FF,SI,CH",3,0.701,9,18930
8,"deGrom, Jacob","FF,SL,CH,SI,CU",3,0.692,10,17403
14,"Glasnow, Tyler","FF,CU,SL,SI,CH",4,0.678,10,12565
18,"Almonte, Yency","FF,ST,SL,SI,CH,FC",2,0.677,7,3531
12,"Anderson, Tyler","FF,CH,FC,SI,CU,SL",6,0.674,10,19608


In [ ]:
df = clean_data('pitcher_data_detailed_cleaned.csv')
print(df['player_name'].unique())
print(df.shape)


['Kershaw, Clayton' 'Sale, Chris' 'Gibson, Kyle' 'Verlander, Justin'
 'Scherzer, Max' 'Nola, Aaron' 'Walker, Taijuan' 'Cole, Gerrit'
 'deGrom, Jacob' 'Gausman, Kevin' 'Manaea, Sean' 'Darvish, Yu'
 'Anderson, Tyler' 'Lugo, Seth' 'Glasnow, Tyler' 'Wheeler, Zack'
 'Buehler, Walker' 'Mikolas, Miles' 'Almonte, Yency' 'Burnes, Corbin'
 'Suarez, Ranger' 'Cease, Dylan' 'Webb, Logan' 'Singer, Brady'
 'Skubal, Tarik' 'McKenzie, Triston' 'Houck, Tanner' 'Crochet, Garrett'
 'Gilbert, Logan' 'Ober, Bailey' 'Sánchez, Cristopher' 'Detmers, Reid'
 'Sears, JP' 'Kirby, George' 'Pepiot, Ryan' 'Miller, Mason'
 'Miller, Bryce' 'Pfaadt, Brandon' 'Skenes, Paul']
(556513, 21)
